## base model

In [1]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv('D:\ML_Projects\Loan_Approval\data\cleaned_loan_data.csv')


In [3]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]
numerical_cols = X.select_dtypes(include='number').columns


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

baseline_score = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)
print(baseline_score)
print("baseline F1 score mean",baseline_score.mean())
print("baseline F1 score std",baseline_score.std())

[0.81842457 0.83372365 0.82472138 0.82552614 0.81695828]
baseline F1 score mean 0.8238708023572483
baseline F1 score std 0.005966351957221804


## Experiment 1: Applying chi2 analysis to remove columns.  

In [4]:
df_exp1 = df.copy()
df_exp1

,person_age,person_gender,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,...,person_education_High School,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22.0,0,71948.0,0,35000.0,16.02,0.49,3.0,561,0,...,0,1,0,0,1,0,0,0,1,0
1,21.0,0,12282.0,0,1000.0,11.14,0.08,2.0,504,1,...,1,0,0,1,0,1,0,0,0,0
2,25.0,0,12438.0,3,5500.0,12.87,0.44,3.0,635,0,...,1,0,0,0,0,0,0,1,0,0
3,23.0,0,79753.0,0,35000.0,15.23,0.44,2.0,675,0,...,0,0,0,0,1,0,0,1,0,0
4,24.0,1,66135.0,1,35000.0,14.27,0.53,4.0,586,0,...,0,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44988,27.0,1,47971.0,6,15000.0,15.66,0.31,3.0,645,0,...,0,0,0,0,1,0,0,1,0,0
44989,37.0,0,65800.0,17,9000.0,14.07,0.14,11.0,621,0,...,0,0,0,0,1,0,1,0,0,0
44990,33.0,1,56942.0,7,2771.0,10.02,0.05,10.0,668,0,...,0,0,0,0,1,0,0,0,0,0
44991,29.0,1,33164.0,4,12000.0,13.23,0.36,6.0,604,0,...,0,0,0,0,1,1,0,0,0,0


---->Removing features<----

In [5]:
chi2_failed_features = ['person_gender','person_education_High School','person_education_Master']
X = df_exp1.drop(chi2_failed_features + ['loan_status'],axis = 1)
y = df_exp1['loan_status']


---->splitting and scaling<----

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
numerical_cols = X.select_dtypes(include='number').columns
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model_exp1 = RandomForestClassifier(random_state=42)

exp1_score = cross_val_score(
    model_exp1,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print(exp1_score)
print("exp1 F1 score mean ",exp1_score.mean())
print("exp1 F1 score mean ",exp1_score.std())


[0.8216072  0.83493856 0.83238159 0.83163612 0.81531229]
exp1 F1 score mean  0.8271751520244444
exp1 F1 score mean  0.007470036989994656


## Experiment 2 : Removing highly correlated features.

The highly correlated features are :  
person_emp_exp <-> person_age  
person_emp_exp <-> cb_person_cred_hist_length   
person_age <-> cb_person_cred_hist_length  

---->Removing person age<----

In [7]:
df_exp2_corr1 = df.copy()
df_exp2_corr1 = df_exp2_corr1.drop('person_age',axis=1)

In [8]:
X = df_exp2_corr1.drop('loan_status',axis=1)
y = df_exp2_corr1['loan_status']
numerical_cols = X.select_dtypes(include="number").columns

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

exp2_corr1 = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print(exp2_corr1)
print("Mean F1:", exp2_corr1.mean())
print("Std:", exp2_corr1.std())

[0.81456732 0.83210702 0.82249322 0.83016304 0.81802893]
Mean F1: 0.8234719085174131
Std: 0.006770701519518838


---->Removing person employement experience<----

In [10]:
df_exp2_corr2 = df.copy()
df_exp2_corr2 = df_exp2_corr2.drop('person_emp_exp',axis=1)

In [11]:
X = df_exp2_corr2.drop('loan_status',axis=1)
y = df_exp2_corr2['loan_status']
numerical_cols = X.select_dtypes(include="number").columns

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

exp2_corr2 = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print(exp2_corr2)
print("Mean F1:", exp2_corr2.mean())
print("Std:", exp2_corr2.std())

[0.81866667 0.82735501 0.82935039 0.8316899  0.81790643]
Mean F1: 0.8249936791361623
Std: 0.005650776799247915


---->Removing cb_person_cred_hist_length<----

In [13]:
df_exp2_corr3 = df.copy()
df_exp2_corr3 = df_exp2_corr3.drop('cb_person_cred_hist_length',axis=1)

In [14]:
X = df_exp2_corr3.drop('loan_status',axis=1)
y = df_exp2_corr3['loan_status']
numerical_cols = X.select_dtypes(include='number').columns

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

exp2_corr3 = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print(exp2_corr3)
print("Mean F1:", exp2_corr3.mean())
print("Std:", exp2_corr3.std())

[0.8185457  0.82793184 0.83091461 0.82304807 0.81726231]
Mean F1: 0.8235405062883878
Std: 0.005258663023108333


## Experiment 3 : Removing chi2 features and person_emp_exp(as it was best among the highly correlated features)

In [16]:
df_exp3 = df.copy()
chi2_failed_features = ['person_gender','person_education_High School','person_education_Master']
X = df_exp3.drop(chi2_failed_features + ['person_emp_exp'] +['loan_status'],axis = 1)
y = df_exp3['loan_status']
numerical_cols = X.select_dtypes(include='number').columns
print(X.columns.tolist())

['person_age', 'person_income', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'person_education_Bachelor', 'person_education_Doctorate', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,stratify=y)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = RandomForestClassifier(random_state=42)

exp3_score = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print(exp3_score)
print("Mean F1:", exp3_score.mean())
print("Std:", exp3_score.std())

[0.82282681 0.83578104 0.83243968 0.83401084 0.81891348]
Mean F1: 0.8287943697789363
Std: 0.006671689039005118


In [21]:
df_feature_engineered = X.copy()
df_feature_engineered["loan_status"] = y
df_feature_engineered.to_csv("feature_engineered_data.csv", index=False)

Conclusion: After multiple feature engineering experiments, the best-performing dataset was obtained by:

Removing Chi² insignificant features.  
Removing person_emp_exp based on correlation analysis.  
This achieved a cross-validated F1 score of 0.8288, improving over the baseline (0.8239).  